In [3]:
import os
import torch
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage import io
from PIL import Image
import torchstain
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
IMAGE_DIR = "GBM_0067_0108"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
UNET_MODEL_PATH = "unet_finetuned_gbm.pth"
TEMP_MASK_DIR = "unet_masks_output_gbm"
CSV_PATH = "xception_patch_mask_features_gbm.csv"

os.makedirs(TEMP_MASK_DIR, exist_ok=True)

# ==========================================
# 🏗️ STEP 1: GENERATE MASKS USING UNET
# ==========================================
print("🚀 Phase 1: Generating U-Net Masks...")

# Load fine-tuned U-Net
unet_model = smp.Unet("resnet34", in_channels=3, classes=1).to(DEVICE)
unet_model.load_state_dict(torch.load(UNET_MODEL_PATH, map_location=DEVICE))
unet_model.eval()

# Simple transform for U-Net inference
unet_tfms = A.Compose([A.Normalize(), ToTensorV2()])

class InferenceDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.fnames = sorted([f for f in os.listdir(img_dir) if f.endswith(('.png', '.jpg', '.tif'))])
        self.transform = transform
    def __len__(self): return len(self.fnames)
    def __getitem__(self, idx):
        fn = self.fnames[idx]
        img = io.imread(os.path.join(self.img_dir, fn))
        if img.ndim == 2: img = np.stack([img]*3, axis=-1)
        if img.shape[-1] == 4: img = img[..., :3]
        if self.transform:
            aug = self.transform(image=img)
            img = aug["image"]
        return img, fn

unet_loader = DataLoader(InferenceDataset(IMAGE_DIR, unet_tfms), batch_size=1)

with torch.no_grad():
    for imgs, fnames in tqdm(unet_loader, desc="U-Net Inference"):
        imgs = imgs.to(DEVICE)
        preds = torch.sigmoid(unet_model(imgs))
        mask_np = (preds[0, 0] > 0.5).cpu().numpy().astype(np.uint8) * 255
        io.imsave(os.path.join(TEMP_MASK_DIR, fnames[0]), mask_np, check_contrast=False)

# ==========================================
# 🧠 PHASE 2: EXTRACT XCEPTION FEATURES
# ==========================================
print("\n🚀 Phase 2: Extracting Xception Features...")

# Load Xception
xception_model = timm.create_model("xception", pretrained=True, num_classes=0, global_pool="avg")
xception_model.eval().to(DEVICE)

# Xception Transforms
xception_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PatchMaskDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.files = sorted(os.listdir(mask_dir)) # Ensure masks exist

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(os.path.join(self.img_dir, fname)).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, fname)).convert("RGB")
        return self.transform(img), self.transform(mask), fname

extract_loader = DataLoader(PatchMaskDataset(IMAGE_DIR, TEMP_MASK_DIR, xception_tfms), 
                            batch_size=32, shuffle=False)

header_written = os.path.exists(CSV_PATH)



with torch.no_grad():
    for imgs, masks, fnames in tqdm(extract_loader, desc="Feature Extraction"):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        
        img_feats = xception_model(imgs).cpu().numpy()
        mask_feats = xception_model(masks).cpu().numpy()

        rows = []
        for i in range(len(fnames)):
            concat_feat = np.concatenate([img_feats[i], mask_feats[i]])
            row = {"filename": fnames[i]}
            for j, v in enumerate(concat_feat):
                row[f"f{j}"] = v
            rows.append(row)

        df = pd.DataFrame(rows)
        df.to_csv(CSV_PATH, mode="a", header=not header_written, index=False)
        header_written = True

print(f"\n✅ All Done! Features saved to: {CSV_PATH}")

🚀 Phase 1: Generating U-Net Masks...


U-Net Inference: 100%|██████████| 98705/98705 [22:22<00:00, 73.54it/s] 
/home/pathouser1/.cellpose/.venv/lib/python3.12/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(



🚀 Phase 2: Extracting Xception Features...


Feature Extraction: 100%|██████████| 3085/3085 [17:41<00:00,  2.91it/s]


✅ All Done! Features saved to: xception_patch_mask_features_gbm.csv
